# Run training on Colab

This notebook runs **the same commands** your AI agents use on your laptop, but on a Colab GPU.
It does not contain any model code. That lives in `src/`.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better), if your run needs a GPU.

Then edit the settings in step 1 and run all cells (Runtime → Run all).

In [ ]:
# @title 1. Settings: edit these
# Where is your project?
#   "drive"  : the project folder is in your Google Drive (recommended; with Google Drive for
#              desktop it syncs to your laptop, so your AI agents see the results automatically)
#   "github" : clone the project from GitHub; results are downloaded as a zip at the end
PROJECT_SOURCE = "drive"  # @param ["drive", "github"]
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/my-research-project"  # @param {type:"string"}
GITHUB_URL = ""  # @param {type:"string"}

# Which experiment to run (the AI tells you which config and extra arguments to use)
CONFIG = "configs/example.yaml"  # @param {type:"string"}
EXTRA_ARGS = ""  # @param {type:"string"}
# Examples of EXTRA_ARGS:  --smoke     --seed 0     --set optim.lr=0.005
RUN_PREPARE_DATA = False  # @param {type:"boolean"}

In [ ]:
# @title 2. Open the project
import os
if PROJECT_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT_PATH
    assert os.path.isdir(PROJECT_DIR), f"Folder not found: {PROJECT_DIR}. Check DRIVE_PROJECT_PATH."
else:
    assert GITHUB_URL, "Set GITHUB_URL in step 1 (private repos: see README, 'Colab with a private repo')."
    PROJECT_DIR = "/content/project"
    if os.path.isdir(PROJECT_DIR):
        !git -C {PROJECT_DIR} pull
    else:
        !git clone {GITHUB_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

In [ ]:
# @title 3. Install packages (Colab already has PyTorch)
!pip install -q -r requirements.txt
import torch, torch_geometric
print("PyTorch", torch.__version__, "| PyG", torch_geometric.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. If this run needs one: Runtime → Change runtime type → T4 GPU, then run all again.")

In [ ]:
# @title 4. Prepare data (only if step 1 says so; splits that already exist are reused)
if RUN_PREPARE_DATA:
    !python -m src.prepare_data --config {CONFIG}
else:
    print("Skipped. Set RUN_PREPARE_DATA = True in step 1 if data/processed/ is empty.")

In [ ]:
# @title 5. Train
import time
START = time.time()
!python -m src.train --config {CONFIG} {EXTRA_ARGS}

In [ ]:
# @title 6. Results of this session
import sys
sys.path.insert(0, PROJECT_DIR)
from src import runlog
new = [r for r in runlog.list_runs(status=None, include_smoke=True)
       if os.path.getmtime(r["dir"]) >= START - 5]
for r in new:
    print(f"{r['status']:9} {r['run_id']}\n          {r['metrics']}")
if not new:
    print("No runs were written. Check the output of step 5 for errors.")

In [ ]:
# @title 7. GitHub mode only: download the new runs
if PROJECT_SOURCE == "github" and new:
    import shutil, tempfile
    from google.colab import files
    tmp = tempfile.mkdtemp()
    for r in new:
        rel = os.path.relpath(r["dir"], PROJECT_DIR)
        shutil.copytree(r["dir"], os.path.join(tmp, rel))
    archive = shutil.make_archive("/content/new_runs", "zip", tmp)
    files.download(archive)
    print("Unzip new_runs.zip inside your project folder on your laptop; it contains runs/<run_id>/ folders.")
else:
    print("Drive mode: results are already in your project's runs/ folder.")